# Chicago Crime Data - Understanding

Step-by-step analysis to understand the dataset, decide which columns to keep or add, and prepare for cleaning and modeling.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.precision', 2)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Load the Chicago crimes dataset
df = pd.read_csv('../data/raw/crimes.csv')

In [ ]:
# Initial glimpse of the data
df.head(10)

## 1. Dataset Overview

Understand the basic structure: shape, columns, data types.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names and types:")
print(df.dtypes)
print("\nBasic stats:")
df.info()

## 2. Missing Values Analysis

Identify columns with missing data - critical for deciding column retention and cleaning strategy.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
missing_df

## 3. Primary Type of Crime

Understand the distribution of crime types - key for predicting "what kind of crimes are more likely."

In [ ]:
primary_counts = df['Primary Type'].value_counts()
print("Primary Type - Top 15:")
print(primary_counts.head(15).to_string())
print("\nTotal unique primary types:", df['Primary Type'].nunique())

In [ ]:
# Visualize primary crime types
import matplotlib.pyplot as plt
primary_counts.head(15).plot(kind='barh', figsize=(10, 6), title='Top 15 Primary Crime Types')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## 4. Location Description (Where crimes occur)

Understand where crimes typically happen - relevant for location-based predictions.

In [ ]:
loc_counts = df['Location Description'].value_counts(dropna=False)
print("Location Description - Top 20:")
print(loc_counts.head(20).to_string())
print("\nTotal unique location types:", df['Location Description'].nunique())

## 5. Geographic Columns (for Hotspot Analysis)

**Beat** is the primary unit for police deployment – patrol areas where officers are assigned. Use Beat (not District) for location predictions. District, Ward, Community Area provide coarser geography. Lat/Lon enable spatial binning.

In [ ]:
print("District - unique count:", df['District'].nunique(), "| Sample:", sorted(df['District'].unique())[:10])
print("Ward - unique count:", df['Ward'].nunique(), "| Range:", df['Ward'].min(), "-", df['Ward'].max())
print("Community Area - unique count:", df['Community Area'].nunique(), "| Missing:", df['Community Area'].isnull().sum())
print("Beat - unique count:", df['Beat'].nunique())
print("\nLatitude/Longitude coverage:", df['Latitude'].notna().sum(), "/", len(df))

## 6. Date/Time Column Analysis

The Date column is critical - we'll extract year, month, day of week, hour, and other temporal features.

In [ ]:
# Parse Date to verify we can extract temporal features
df_temp = df.copy()
df_temp['Date_parsed'] = pd.to_datetime(df_temp['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
print("Sample parsed dates:")
print(df_temp[['Date', 'Date_parsed']].dropna().head())
print("\nParse errors (null):", df_temp['Date_parsed'].isnull().sum())

## 7. Categorical and Binary Columns

Arrest, Domestic, FBI Code - useful for modeling and EDA.

In [ ]:
print("Arrest distribution:", df['Arrest'].value_counts())
print("\nDomestic distribution:", df['Domestic'].value_counts())
print("\nFBI Code - top 10:", df['FBI Code'].value_counts().head(10).to_string())

## 8. Duplicate and Redundant Columns

Identify columns that may be redundant (e.g., Location vs Lat/Lon, Year in Date).

In [ ]:
# Location is (Latitude, Longitude) - redundant
print("Location sample:", df['Location'].iloc[0])
print("Latitude, Longitude:", df['Latitude'].iloc[0], df['Longitude'].iloc[0])
print("\nDuplicate rows (by ID):", df.duplicated(subset=['ID']).sum())
print("Duplicate Case Numbers:", df['Case Number'].duplicated().sum())

## 9. Column Retention & Addition Recommendations

**Columns to KEEP:**
- **ID** – Unique identifier (useful for tracking)
- **Date** – Source for temporal features
- **Block** – Street-level location context
- **Primary Type** – Main crime type (key target for prediction)
- **Location Description** – Where crime occurred
- **Arrest, Domestic** – Binary outcomes/features
- **Beat** – Primary unit for police deployment (preferred over District for location prediction)
- **District, Ward, Community Area** – Geographic hierarchy, coarser than Beat
- **Latitude, Longitude** – For mapping and spatial modeling
- **Year** – Already extracted (or derive from Date)
- **IUCR, Description, FBI Code** – Optional for richer crime categorization

**Columns to DROP:**
- **Case Number** – Administrative; ID is sufficient
- **Location** – Redundant with Latitude/Longitude
- **X Coordinate, Y Coordinate** – State plane coordinates; Lat/Lon preferred for mapping
- **Updated On** – Metadata, not predictive

**Columns to ADD (from Date):**
- **Month** (1–12)
- **DayOfWeek** (0–6 or name)
- **Hour** (0–23)
- **Quarter** (1–4)
- **WeekOfYear** (optional)
- **IsWeekend** (boolean)

**Columns to ADD (for modeling):**
- **Lat_bin, Lon_bin** – Spatial grid (0.02 deg) for Beat prediction
- **HourBin** – 6 time-of-day bins
- **Crime_Category** – 5 groups (Violent, Theft, Property, Narcotics, Other)

**Aggregation for modeling:** Aggregate by (Beat, WeekOfYear, DayOfWeek, HourBin) to predict crime levels per bin.

In [ ]:
# Summary of recommended columns and additions
recommendations = """
FINAL COLUMN DECISIONS:
======================
KEEP: ID, Date, Block, IUCR, Primary Type, Description, Location Description,
      Arrest, Domestic, Beat, District, Ward, Community Area, FBI Code,
      Latitude, Longitude, Year

DROP: Case Number, Location, X Coordinate, Y Coordinate, Updated On

ADD (from Date): Month, DayOfWeek, Hour, Quarter, WeekOfYear, IsWeekend
"""
print(recommendations)